# 🎭 ChuckleNet: Process Local Files (FAST VERSION)

**Key fix**: Use ffmpeg for fast segment extraction instead of librosa.load()
**Speed**: ~20 min total vs 95 hours

**Before running**: Files must be uploaded to Google Drive:
```bash
rclone copy /Users/Subho/data/utterances/vtt_audio_local/ gdrive:chuckle_net/audio/
rclone copy /Users/Subho/data/chuckle_vtt_labels/ gdrive:chuckle_net/vtt/
```

In [ ]:
# 1. Setup
!apt-get install -y ffmpeg 2>&1 | tail -3
!pip install librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3

from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import subprocess
from tqdm import tqdm
import re

# Find Drive mount
for BASE in ['/content/drive/My Drive/chuckle_net',
              '/content/drive/Shareddrives/chuckle_net',
              '/content/drive/MyDrive/chuckle_net']:
    if os.path.exists(BASE):
        print(f'✅ Found: {BASE}')
        break

AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

audio_files = glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.wav')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')

print(f'Audio: {len(audio_files)} files')
print(f'VTT: {len(vtt_files)} files')
print(f'Sample audio: {os.path.basename(audio_files[0]) if audio_files else "none"}')
print(f'Sample VTT: {os.path.basename(vtt_files[0]) if vtt_files else "none"}')

In [ ]:
# 2. Build audio → VTT mapping (FAST)
def get_vid(name):
    return name.replace('.en.vtt','').replace('.vtt','').replace('.m4a','').replace('.wav','').replace('.mp3','')

vtt_lookup = {get_vid(os.path.basename(v)): v for v in vtt_files}
audio_lookup = {get_vid(os.path.basename(a)): a for a in audio_files}

matching = set(audio_lookup.keys()) & set(vtt_lookup.keys())
print(f'Matching pairs: {len(matching)}')

if matching:
    vid = list(matching)[0]
    print(f'Sample: {vid}')
    print(f'  Audio: {os.path.basename(audio_lookup[vid])}')
    print(f'  VTT: {os.path.basename(vtt_lookup[vid])}')

In [ ]:
# 3. Parse VTT (fast)
def parse_vtt(vtt_path):
    """Return list of (start_sec, end_sec, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    def to_sec(ts):
        p = ts.replace('.',':').split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    
    cues, lines = [], content.split('\n')
    i = 0
    while i < len(lines):
        if '-->' in lines[i]:
            s, e = lines[i].split('-->')
            s, e = to_sec(s.strip()), to_sec(e.strip())
            txt, i = [], i+1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                txt.append(lines[i].strip()); i += 1
            text = ' '.join(txt)
            cues.append((s, e, text, '[laughter]' in text.lower()))
        else: i += 1
    return cues

# 4. Extract F0 using ffmpeg + librosa on SMALL chunks only
# Key insight: extract ONLY the needed segment, not whole file
import tempfile
import librosa

def extract_segment_f0(audio_path, start, end, sr=22050):
    """Extract F0 features from audio segment using ffmpeg + librosa."""
    duration = end - start
    if duration <= 0 or duration > 30:
        return None
    
    # Use ffmpeg to extract segment (fast!)
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    
    try:
        # ffmpeg extracts segment directly - very fast
        cmd = [
            'ffmpeg', '-y', '-ss', str(start), '-t', str(duration),
            '-i', audio_path, '-ar', str(sr), '-ac', '1', '-loglevel', 'error',
            tmp_path
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=10)
        
        if result.returncode != 0:
            return None
        
        # Load the small segment with librosa (fast since file is tiny)
        y, _ = librosa.load(tmp_path, sr=sr)
        
        if len(y) < sr * 0.05:  # < 50ms
            return None
        
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=512)
        f0 = np.nan_to_num(f0, nan=0.0)
        
        return [
            float(np.mean(f0)), float(np.std(f0)),
            float(np.max(f0)), float(np.min(f0)),
            float(np.mean(f0 > 0))
        ]
    except:
        return None
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

# Test speed
import time
if matching:
    vid = list(matching)[0]
    cues = parse_vtt(vtt_lookup[vid])
    print(f'Test: {vid}, {len(cues)} cues')
    
    t0 = time.time()
    seg = extract_segment_f0(audio_lookup[vid], cues[0][0], cues[0][1])
    t1 = time.time()
    print(f'Single segment extraction: {t1-t0:.2f}s')
    print(f'Feature: {seg}')
    
    # Estimate total time
    avg_cues = 100  # average cues per video
    est_per_video = (t1-t0) * avg_cues
    est_total = est_per_video * len(matching) / 60
    print(f'Estimated per video: {est_per_video:.0f}s')
    print(f'Estimated total: {est_total:.0f} min for {len(matching)} videos')

In [ ]:
# 5. Process ALL videos (with progress)
all_features, all_labels, all_vids, all_langs = [], [], [], []
failed = 0

matching_list = list(matching)
print(f'Processing {len(matching_list)} videos...')
print(f'Estimated time: ~15-20 min total')

for vid in tqdm(matching_list, desc='Videos'):
    audio_path = audio_lookup[vid]
    vtt_path = vtt_lookup[vid]
    
    # Language from VTT filename
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    try:
        cues = parse_vtt(vtt_path)
        
        for i, (start, end, text, has_laughter) in enumerate(cues):
            feat = extract_segment_f0(audio_path, start, end)
            if feat is not None:
                all_features.append(feat)
                all_labels.append(1 if has_laughter else 0)
                all_vids.append(vid)
                all_langs.append(lang)
    except Exception as e:
        failed += 1

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)
langs = np.array(all_langs)

print(f'\n✅ Done! Processed: {len(X)} segments')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')
print(f'Failed videos: {failed}')

for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% positive')

In [ ]:
# 6. Train + Evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

unique_vids = list(set(vids))
np.random.seed(42); np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids)*0.2))
test_vids = set(unique_vids[:n_test])

X_train = np.array([X[i] for i in range(len(vids)) if vids[i] not in test_vids])
y_train = np.array([y[i] for i in range(len(vids)) if vids[i] not in test_vids])
X_test = np.array([X[i] for i in range(len(vids)) if vids[i] in test_vids])
y_test = np.array([y[i] for i in range(len(vids)) if vids[i] in test_vids])

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)')
print(f'Test: {len(X_test)} ({100*y_test.mean():.1f}% pos)\n')

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print('=== Logistic Regression ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print('\n=== MLP ===')
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 7. Save
import pickle
out = {'features': X, 'labels': y, 'vids': vids, 'langs': langs}
np.savez_compressed(f'{BASE}/processed_620.npz', **out)
with open(f'{BASE}/f0_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Saved: {BASE}/processed_620.npz')
print(f'Model: {BASE}/f0_model.pkl')
print('\n🎉 DONE!')